# The Battle Class
This class is for making a Battle object.  A battle object takes a file name (formatted as a string, assuming a file with that name exists in data/replays/gen9-randombattle) as input.  From that string, it will gather the information of usernames, pre-battle ratings, post-battle ratings (if the match was rated), pokemon on each team, battle start and end time, and winner.  It has a nice string representation.

Next goal: figure out how to take the pokemon names and access information like type and stats

In [1]:
import json

class Battle:
    def __init__(self,file_name):
        with open("replays/gen9-randombattle/" + file_name,"r") as battle_json:
            data = json.load(battle_json)
        self.log = data["log"]
        self.player_names = [data["players"][0],data["players"][1]]
        self.old_player_ratings = {} # a dictionary where the keys are the player usernames and the values are the player ratings at the start of the match
        self.new_player_ratings = {} # a dictionary where the keys are the player usernames and the values are the player ratings after the end of the match
        self.winner = ""
        self.loser = ""
        self.lead_pokemon = {self.player_names[0] : "", self.player_names[1] : ""} # dictionary where keys are player usernames and values are the first pokemon each player sent onto the field
        self.teams = {self.player_names[0] : set(), self.player_names[1]: set()} # a dictionary where the keys are player usernames and the values are sets containing the pokemon names
        self.time_list = [] # for keeping track of all move times, gets thrown away now.  Could be used to check for time-out forfeits?
        self.start_time = 0
        self.end_time = 0
        self.rated = (data["rating"] != None)
        self.log_parser(self.log)

    def __repr__(self):
        to_return = ""
        if self.rated:
            to_return += f"This was a battle between {self.player_names[0]} (pre-match rating {self.old_player_ratings[self.player_names[0]]}) and {self.player_names[1]} (pre-match rating {self.old_player_ratings[self.player_names[1]]}).\n"
        else:
            to_return += f"This was a battle between {self.player_names[0]} and {self.player_names[1]}.\n"
        for index in range(2):
            to_return += f"{self.player_names[index]}'s team was {self.teams[self.player_names[index]]} and their lead pokemon was {self.lead_pokemon[self.player_names[index]]}.\n"
        to_return += f"{self.winner} won!\n"
        if self.rated:
            to_return += f"{self.winner}'s rating increased to {self.new_player_ratings[self.winner]}.\n"
            to_return += f"{self.loser}'s rating fell to {self.new_player_ratings[self.loser]}."
        else:
            to_return += "This was an unrated match, so no one's rating changed."
        return to_return

    def log_parser(self,log):
        # This loop generates battle information by searching for *all* instances of "|t:|", "|switch|", "|win|", and " for winning)"
        # In particular, it sets the Battle object's new_player_ratings, winner, loser, teams, time_list, and rated attributes.
        for i in range(len(log)):
            # generates time list
            if log.startswith("|t:|",i):
                j = i + 4
                time_string = ""
                while log[j] != "\n":
                    time_string += log[j]
                    j += 1
                self.time_list.append(int(time_string))
            # generate team list
            if log.startswith("|switch|",i):
                pokemon_name = ""
                # Skip to the third pipe in the line.  This is where the species (including forme) is.
                pipe_counter = 0
                j = i
                while pipe_counter < 3:
                    if log[j] == "|":
                        pipe_counter += 1
                    j += 1
                # Now j is the first index after the third pipe in the line
                while log[j] != ",":
                    pokemon_name += log[j]
                    j += 1
                if log.startswith("p1a",i+8):
                    player_name = self.player_names[0]
                else:
                    player_name = self.player_names[1]
                self.teams[player_name].add(pokemon_name)
                if self.lead_pokemon[player_name] == "": # identifies  
                    self.lead_pokemon[player_name] = pokemon_name
            # identifies winner and loser
            if log.startswith("|win|",i):
                j = i + 5
                while log[j] != "\n":
                    self.winner += log[j]
                    j += 1
                winner_index = self.player_names.index(self.winner)
                self.loser = self.player_names[1-winner_index]
            if self.rated:
                if log.startswith(" for losing)",i):
                    delta_index = i-1
                    score_delta_string = ""
                    while not log[delta_index] in ["-","+"]: # if a player's rating changes by 0 after losing, the log will write +0 instead of -0 for the loser
                        score_delta_string = log[delta_index] + score_delta_string
                        delta_index -= 1
                    score_delta = int(score_delta_string)
                    new_score_index = delta_index - 17
                    new_score_string = ""
                    while log[new_score_index] != ">":
                        new_score_string = log[new_score_index] + new_score_string
                        new_score_index -= 1
                    new_score = int(new_score_string)
                    old_score = new_score + score_delta
                    self.old_player_ratings[self.loser] = old_score
                    self.new_player_ratings[self.loser] = new_score
                if log.startswith(" for winning)",i):
                    delta_index = i-1
                    score_delta_string = ""
                    while not log[delta_index] in ["+","-"]:
                        score_delta_string = log[delta_index] + score_delta_string
                        delta_index -= 1
                    score_delta = int(score_delta_string)
                    new_score_index = delta_index - 17
                    new_score_string = ""
                    while log[new_score_index] != ">":
                        new_score_string = log[new_score_index] + new_score_string
                        new_score_index -= 1
                    new_score = int(new_score_string)
                    old_score = new_score - score_delta
                    self.old_player_ratings[self.winner] = old_score
                    self.new_player_ratings[self.winner] = new_score
        # We separate start and end time for ease of access
        self.start_time = self.time_list[0]
        self.end_time = self.time_list[-1]

## Examples
Here, you can see how the Battle class functions on a couple of the files in data/replays/gen9-randombattle.  It's pretty straightforward.

In [2]:
battle1 = Battle("gen9randombattle-2631360263.json")
battle1

This was a battle between LaxMD (pre-match rating 1789) and N.TdaRajada (pre-match rating 1837).
LaxMD's team was {'Delphox', 'Tauros-Paldea-Aqua', 'Wugtrio', 'Victreebel', 'Indeedee-F', 'Skarmory'} and their lead pokemon was Delphox.
N.TdaRajada's team was {'Torterra', 'Snorlax', 'Wigglytuff', 'Lokix', 'Mandibuzz', 'Mienshao'} and their lead pokemon was Wigglytuff.
N.TdaRajada won!
N.TdaRajada's rating increased to 1854.
LaxMD's rating fell to 1772.

In [3]:
battle2 = Battle("gen9randombattle-2631363920.json")
battle2

This was a battle between kaisarian and Flamesenpai557.
kaisarian's team was {'Pincurchin', 'Ambipom', 'Mimikyu', 'Breloom', 'Overqwil', 'Brute Bonnet'} and their lead pokemon was Mimikyu.
Flamesenpai557's team was {'Arceus-Ghost', 'Kilowattrel', 'Gurdurr', 'Dugtrio-Alola', 'Slaking', 'Breloom'} and their lead pokemon was Kilowattrel.
kaisarian won!
This was an unrated match, so no one's rating changed.

In [4]:
battle3 = Battle("gen9randombattle-2631365384.json")
battle3

This was a battle between rgrgreger (pre-match rating 1895) and Ismusicalschool (pre-match rating 1946).
rgrgreger's team was {'Primarina', 'Flareon', 'Sudowoodo', 'Electrode-Hisui', 'Donphan'} and their lead pokemon was Electrode-Hisui.
Ismusicalschool's team was {'Houndoom', 'Slowking', 'Lugia', 'Hippowdon', 'Iron Treads', 'Luvdisc'} and their lead pokemon was Iron Treads.
rgrgreger won!
rgrgreger's rating increased to 1918.
Ismusicalschool's rating fell to 1923.

In [5]:
battle4 = Battle("gen9randombattle-2631511033.json")
battle4

This was a battle between MMF6G and MandoWarrior.
MMF6G's team was {'Gyarados', 'Torterra', 'Conkeldurr', 'Solgaleo', 'Rampardos', 'Drifblim'} and their lead pokemon was Solgaleo.
MandoWarrior's team was {'Whimsicott', 'Ting-Lu', 'Smeargle', 'Golduck', 'Tinkaton', 'Dondozo'} and their lead pokemon was Dondozo.
MandoWarrior won!
This was an unrated match, so no one's rating changed.

In [6]:
battle5 = Battle("gen9randombattle-2631533192.json")
battle5

This was a battle between lalo's pollos (pre-match rating 1897) and WhadayatalkinAbout (pre-match rating 1921).
lalo's pollos's team was {'Espathra', 'Stonjourner', 'Fezandipiti', 'Regirock', 'Mienshao', 'Araquanid'} and their lead pokemon was Mienshao.
WhadayatalkinAbout's team was {'Magearna', 'Koraidon', 'Glimmora', 'Copperajah'} and their lead pokemon was Copperajah.
WhadayatalkinAbout won!
WhadayatalkinAbout's rating increased to 1940.
lalo's pollos's rating fell to 1878.

# Sandbox

Below is what I was using to test.  You should probably ignore this stuff.

In [6]:
with open("replays/gen9-randombattle/gen9randombattle-2631360263.json","r") as battle_json:
    data = json.load(battle_json)
log = data["log"]
player_names = [data["players"][0],data["players"][1]]
old_player_ratings = {} # a dictionary where the keys are the player usernames and the values are the player ratings at the start of the match
new_player_ratings = {} # a dictionary where the keys are the player usernames and the values are the player ratings after the end of the match
winner = ""
loser = ""
teams = {player_names[0]: set(), player_names[1]: set()} # a dictionary where the keys are player usernames and the values are sets containing the pokemon names
time_list = [] # for keeping track of all move times, gets thrown away now.  Could be used to check for time-out forfeits?
start_time = 0
end_time = 0

for i in range(len(log)):
    # generates time list
    if log.startswith("|t:",i):
        j = i + 4
        time_string = ""
        while log[j] != "\n":
            time_string += log[j]
            j += 1
        time_list.append(int(time_string))
    # generate team list
    if log.startswith("|switch|",i):
        pokemon_name = ""
        # Skip to the third pipe in the line.  This is where the species (including forme) is.
        pipe_counter = 0
        j = i
        while pipe_counter < 3:
            if log[j] == "|":
                pipe_counter += 1
            j += 1
        # Now j is the first index after the third pipe in the line
        while log[j] != ",":
            pokemon_name += log[j]
            j += 1
        if log.startswith("p1a",i+8):
            player_name = player_names[0]
        else:
            player_name = player_names[1]
        teams[player_name].add(pokemon_name)
    # identifies winner and loser
    if log.startswith("|win|",i):
        j = i + 5
        while log[j] != "\n":
            winner += log[j]
            j += 1
        winner_index = player_names.index(winner)
        loser = player_names[1-winner_index]
    for player_index in range(2):
        if log.startswith(f"{player_names[player_index]}'s rating: ",i):
            j = len(player_names[player_index]) + 11
            old_score_string = ""
            while log[i + j] != " ":
                old_score_string += log[i+j]
                j += 1
            old_player_ratings[player_names[player_index]] = int(old_score_string)
            new_player_ratings[player_names[player_index]] = int(log[i + j + 16:i + j + 16 + len(old_score_string)])
    

start_time = time_list[0]
end_time = time_list[-1]

for index in range(2):
    print(f"{player_names[index]}'s old rating:", old_player_ratings[player_names[index]])
    print(f"{player_names[index]}'s team:", teams[player_names[index]])
print("winner:", winner)
try:
    for index in range(2):
        print(f"{player_names[index]} new rating:", new_player_ratings[player_names[index]])
except KeyError:
    print("This was not a rated match")

print("start time:", start_time)
print("end time:", end_time)

LaxMD's old rating: 1789
LaxMD's team: {'Delphox', 'Indeedee-F', 'Skarmory', 'Victreebel', 'Tauros-Paldea-Aqua', 'Wugtrio'}
N.TdaRajada's old rating: 1837
N.TdaRajada's team: {'Wigglytuff', 'Torterra', 'Snorlax', 'Mienshao', 'Mandibuzz', 'Lokix'}
winner: N.TdaRajada
LaxMD new rating: 1772
N.TdaRajada new rating: 1854
start time: 1781375540
end time: 1781376361


In [164]:
player_name = "LaxMD"
i = log.index(f"{player_name}'s rating: ")
j = len(player_name) + 11
log[i + j]


'1'

In [43]:
len(line)

26

In [136]:
with open("replays/gen9-randombattle/gen9randombattle-2631363920.json","r") as battle_json:
    data = json.load(battle_json)

data["players"][1]

'Flamesenpai557'

In [139]:
log="|j|☆xValk_\n|j|☆manwhocouldntcook\n|t:|1781540286\n|gametype|singles\n|player|p1|xValk_|ltsurge|2138\n|player|p2|manwhocouldntcook|valerie|2146\n|gen|9\n|tier|[Gen 9] Random Battle\n|rated|\n|rule|Species Clause: Limit one of each Pokémon\n|rule|HP Percentage Mod: HP is shown in percentages\n|rule|Sleep Clause Mod: Limit one foe put to sleep\n|rule|Illusion Level Mod: Illusion disguises the Pokémon's true level\n|\n|t:|1781540286\n|teamsize|p1|6\n|teamsize|p2|6\n|start\n|switch|p1a: Ho-Oh|Ho-Oh, L71|268/268\n|switch|p2a: Scrafty|Scrafty, L83, M|244/244\n|turn|1\n|\n|t:|1781540296\n|switch|p2a: Incineroar|Incineroar, L82, F|290/290\n|-ability|p2a: Incineroar|Intimidate|boost\n|-unboost|p1a: Ho-Oh|atk|1\n|move|p1a: Ho-Oh|Brave Bird|p2a: Incineroar\n|-damage|p2a: Incineroar|206/290\n|-damage|p1a: Ho-Oh|240/268|[from] Recoil\n|\n|upkeep\n|turn|2\n|\n|t:|1781540301\n|switch|p1a: Carbink|Carbink, L90|236/236\n|move|p2a: Incineroar|Knock Off|p1a: Carbink\n|-resisted|p1a: Carbink\n|-damage|p1a: Carbink|205/236\n|-enditem|p1a: Carbink|Chesto Berry|[from] move: Knock Off|[of] p2a: Incineroar\n|\n|upkeep\n|turn|3\n|inactive|Battle timer is ON: inactive players will automatically lose when time's up. (requested by manwhocouldntcook)\n|\n|t:|1781540308\n|switch|p2a: Vileplume|Vileplume, L85, F|266/266\n|move|p1a: Carbink|Moonblast|p2a: Vileplume\n|-resisted|p2a: Vileplume\n|-damage|p2a: Vileplume|230/266\n|\n|-heal|p2a: Vileplume|246/266|[from] item: Leftovers\n|upkeep\n|turn|4\n|\n|t:|1781540313\n|switch|p1a: Victreebel|Victreebel, L90, M|290/290\n|move|p2a: Vileplume|Sleep Powder|p1a: Victreebel\n|-immune|p1a: Victreebel\n|\n|-heal|p2a: Vileplume|262/266|[from] item: Leftovers\n|upkeep\n|turn|5\n|\n|t:|1781540316\n|move|p1a: Victreebel|Sunny Day|p1a: Victreebel\n|-weather|SunnyDay\n|move|p2a: Vileplume|Sludge Bomb|p1a: Victreebel\n|-damage|p1a: Victreebel|175/290\n|\n|-weather|SunnyDay|[upkeep]\n|-heal|p2a: Vileplume|266/266|[from] item: Leftovers\n|upkeep\n|turn|6\n|\n|t:|1781540320\n|switch|p2a: Incineroar|Incineroar, L82, F|206/290\n|-ability|p2a: Incineroar|Intimidate|boost\n|-unboost|p1a: Victreebel|atk|1\n|move|p1a: Victreebel|Sludge Wave|p2a: Incineroar\n|-damage|p2a: Incineroar|62/290\n|-damage|p1a: Victreebel|146/290|[from] item: Life Orb\n|\n|-weather|SunnyDay|[upkeep]\n|upkeep\n|turn|7\n|\n|t:|1781540325\n|move|p1a: Victreebel|Sludge Wave|p2a: Incineroar\n|-damage|p2a: Incineroar|0 fnt\n|faint|p2a: Incineroar\n|-damage|p1a: Victreebel|117/290|[from] item: Life Orb\n|\n|-weather|SunnyDay|[upkeep]\n|upkeep\n|\n|t:|1781540335\n|switch|p2a: Scrafty|Scrafty, L83, M|244/244\n|turn|8\n|\n|t:|1781540338\n|move|p1a: Victreebel|Weather Ball|p2a: Scrafty\n|-damage|p2a: Scrafty|121/244\n|-damage|p1a: Victreebel|88/290|[from] item: Life Orb\n|move|p2a: Scrafty|Knock Off|p1a: Victreebel\n|-damage|p1a: Victreebel|0 fnt\n|-enditem|p1a: Victreebel|Life Orb|[from] move: Knock Off|[of] p2a: Scrafty\n|faint|p1a: Victreebel\n|\n|-weather|SunnyDay|[upkeep]\n|-heal|p2a: Scrafty|136/244|[from] item: Leftovers\n|upkeep\n|\n|t:|1781540342\n|switch|p1a: Ho-Oh|Ho-Oh, L71|268/268\n|turn|9\n|\n|t:|1781540347\n|move|p1a: Ho-Oh|Sacred Fire|p2a: Scrafty\n|-damage|p2a: Scrafty|24/244\n|-status|p2a: Scrafty|brn\n|move|p2a: Scrafty|Rest|p2a: Scrafty\n|-status|p2a: Scrafty|slp|[from] move: Rest\n|-heal|p2a: Scrafty|244/244 slp|[silent]\n|\n|-weather|none\n|upkeep\n|turn|10\n|\n|t:|1781540353\n|switch|p2a: Jolteon|Jolteon, L84, F|246/246\n|move|p1a: Ho-Oh|Brave Bird|p2a: Jolteon\n|-resisted|p2a: Jolteon\n|-damage|p2a: Jolteon|169/246\n|-damage|p1a: Ho-Oh|243/268|[from] Recoil\n|\n|-heal|p2a: Jolteon|184/246|[from] item: Leftovers\n|upkeep\n|turn|11\n|\n|t:|1781540357\n|switch|p1a: Carbink|Carbink, L90|205/236\n|move|p2a: Jolteon|Substitute|p2a: Jolteon\n|-start|p2a: Jolteon|Substitute\n|-damage|p2a: Jolteon|123/246\n|\n|-heal|p2a: Jolteon|138/246|[from] item: Leftovers\n|upkeep\n|turn|12\n|\n|t:|1781540362\n|move|p2a: Jolteon|Thunderbolt|p1a: Carbink\n|-damage|p1a: Carbink|141/236\n|move|p1a: Carbink|Body Press|p2a: Jolteon\n|-end|p2a: Jolteon|Substitute\n|\n|-heal|p2a: Jolteon|153/246|[from] item: Leftovers\n|upkeep\n|turn|13\n|\n|t:|1781540368\n|switch|p2a: Vileplume|Vileplume, L85, F|266/266\n|switch|p1a: Ho-Oh|Ho-Oh, L71|268/268\n|\n|upkeep\n|turn|14\n|\n|t:|1781540372\n|move|p1a: Ho-Oh|Sacred Fire|p2a: Vileplume\n|-supereffective|p2a: Vileplume\n|-damage|p2a: Vileplume|68/266\n|-status|p2a: Vileplume|brn\n|move|p2a: Vileplume|Sleep Powder|p1a: Ho-Oh\n|-status|p1a: Ho-Oh|slp|[from] move: Sleep Powder\n|\n|-heal|p2a: Vileplume|84/266 brn|[from] item: Leftovers\n|-damage|p2a: Vileplume|68/266 brn|[from] brn\n|upkeep\n|turn|15\n|\n|t:|1781540377\n|cant|p1a: Ho-Oh|slp\n|move|p2a: Vileplume|Strength Sap|p1a: Ho-Oh\n|-unboost|p1a: Ho-Oh|atk|1\n|-heal|p2a: Vileplume|266/266 brn\n|\n|-damage|p2a: Vileplume|250/266 brn|[from] brn\n|upkeep\n|turn|16\n|\n|t:|1781540380\n|switch|p2a: Scrafty|Scrafty, L83, M|244/244 slp\n|-curestatus|p1a: Ho-Oh|slp|[msg]\n|move|p1a: Ho-Oh|Brave Bird|p2a: Scrafty\n|-supereffective|p2a: Scrafty\n|-damage|p2a: Scrafty|112/244 slp\n|-damage|p1a: Ho-Oh|224/268|[from] Recoil\n|\n|-heal|p2a: Scrafty|127/244 slp|[from] item: Leftovers\n|upkeep\n|turn|17\n|\n|t:|1781540387\n|move|p1a: Ho-Oh|Earthquake|p2a: Scrafty\n|-damage|p2a: Scrafty|92/244 slp\n|cant|p2a: Scrafty|slp\n|\n|-heal|p2a: Scrafty|107/244 slp|[from] item: Leftovers\n|upkeep\n|turn|18\n|\n|t:|1781540390\n|switch|p2a: Vileplume|Vileplume, L85, F|250/266 brn\n|move|p1a: Ho-Oh|Brave Bird|p2a: Vileplume\n|-supereffective|p2a: Vileplume\n|-damage|p2a: Vileplume|88/266 brn\n|-damage|p1a: Ho-Oh|171/268|[from] Recoil\n|\n|-heal|p2a: Vileplume|104/266 brn|[from] item: Leftovers\n|-damage|p2a: Vileplume|88/266 brn|[from] brn\n|upkeep\n|turn|19\n|\n|t:|1781540394\n|move|p1a: Ho-Oh|Sacred Fire|p2a: Vileplume\n|-supereffective|p2a: Vileplume\n|-damage|p2a: Vileplume|0 fnt\n|faint|p2a: Vileplume\n|\n|upkeep\n|\n|t:|1781540396\n|switch|p2a: Jolteon|Jolteon, L84, F|153/246\n|turn|20\n|\n|t:|1781540399\n|switch|p1a: Carbink|Carbink, L90|141/236\n|move|p2a: Jolteon|Substitute|p2a: Jolteon\n|-start|p2a: Jolteon|Substitute\n|-damage|p2a: Jolteon|92/246\n|\n|-heal|p2a: Jolteon|107/246|[from] item: Leftovers\n|upkeep\n|turn|21\n|\n|t:|1781540401\n|move|p2a: Jolteon|Calm Mind|p2a: Jolteon\n|-boost|p2a: Jolteon|spa|1\n|-boost|p2a: Jolteon|spd|1\n|move|p1a: Carbink|Body Press|p2a: Jolteon\n|-end|p2a: Jolteon|Substitute\n|\n|-heal|p2a: Jolteon|122/246|[from] item: Leftovers\n|upkeep\n|turn|22\n|\n|t:|1781540403\n|move|p2a: Jolteon|Thunderbolt|p1a: Carbink\n|-damage|p1a: Carbink|44/236\n|move|p1a: Carbink|Body Press|p2a: Jolteon\n|-damage|p2a: Jolteon|8/246\n|\n|-heal|p2a: Jolteon|23/246|[from] item: Leftovers\n|upkeep\n|turn|23\n|\n|t:|1781540406\n|move|p2a: Jolteon|Thunderbolt|p1a: Carbink\n|-damage|p1a: Carbink|0 fnt\n|faint|p1a: Carbink\n|\n|-heal|p2a: Jolteon|38/246|[from] item: Leftovers\n|upkeep\n|\n|t:|1781540407\n|switch|p1a: Cyclizar|Cyclizar, L83, M|252/252\n|turn|24\n|\n|t:|1781540410\n|-terastallize|p2a: Jolteon|Ice\n|move|p2a: Jolteon|Tera Blast|p1a: Cyclizar|[anim] Tera Blast Ice\n|-supereffective|p1a: Cyclizar\n|-damage|p1a: Cyclizar|0 fnt\n|faint|p1a: Cyclizar\n|\n|-heal|p2a: Jolteon|53/246|[from] item: Leftovers\n|upkeep\n|\n|t:|1781540416\n|switch|p1a: Mewtwo|Mewtwo, L72|272/272\n|-ability|p1a: Mewtwo|Unnerve\n|turn|25\n|\n|t:|1781540423\n|move|p2a: Jolteon|Tera Blast|p1a: Mewtwo|[anim] Tera Blast Ice\n|-damage|p1a: Mewtwo|127/272\n|move|p1a: Mewtwo|Psystrike|p2a: Jolteon\n|-damage|p2a: Jolteon|0 fnt\n|faint|p2a: Jolteon\n|-damage|p1a: Mewtwo|100/272|[from] item: Life Orb\n|\n|upkeep\n|\n|t:|1781540429\n|switch|p2a: Staraptor|Staraptor, L79, M|263/263\n|turn|26\n|\n|t:|1781540433\n|move|p1a: Mewtwo|Psystrike|p2a: Staraptor\n|-damage|p2a: Staraptor|68/263\n|-damage|p1a: Mewtwo|73/272|[from] item: Life Orb\n|move|p2a: Staraptor|U-turn|p1a: Mewtwo\n|-supereffective|p1a: Mewtwo\n|-damage|p1a: Mewtwo|0 fnt\n|faint|p1a: Mewtwo\n|\n|t:|1781540436\n|switch|p2a: Scrafty|Scrafty, L83, M|107/244 slp|[from] U-turn\n|\n|-heal|p2a: Scrafty|122/244 slp|[from] item: Leftovers\n|upkeep\n|\n|t:|1781540440\n|switch|p1a: Ho-Oh|Ho-Oh, L71|260/268\n|turn|27\n|\n|t:|1781540448\n|move|p1a: Ho-Oh|Brave Bird|p2a: Scrafty\n|-supereffective|p2a: Scrafty\n|-damage|p2a: Scrafty|0 fnt\n|faint|p2a: Scrafty\n|-damage|p1a: Ho-Oh|220/268|[from] Recoil\n|\n|upkeep\n|\n|t:|1781540452\n|switch|p2a: Staraptor|Staraptor, L79, M|68/263\n|turn|28\n|\n|t:|1781540460\n|move|p2a: Staraptor|Brave Bird|p1a: Ho-Oh\n|-damage|p1a: Ho-Oh|0 fnt\n|faint|p1a: Ho-Oh\n|-damage|p2a: Staraptor|0 fnt|[from] Recoil\n|faint|p2a: Staraptor\n|\n|upkeep\n|\n|t:|1781540462\n|switch|p2a: Hitmonlee|Hitmonlee, L85, M|224/224\n|switch|p1a: Lugia|Lugia, L73|275/275\n|turn|29\n|\n|t:|1781540468\n|move|p1a: Lugia|Aeroblast|p2a: Hitmonlee\n|-supereffective|p2a: Hitmonlee\n|-damage|p2a: Hitmonlee|102/224\n|move|p2a: Hitmonlee|Knock Off|p1a: Lugia\n|-supereffective|p1a: Lugia\n|-damage|p1a: Lugia|208/275\n|-enditem|p1a: Lugia|Heavy-Duty Boots|[from] move: Knock Off|[of] p2a: Hitmonlee\n|\n|upkeep\n|turn|30\n|\n|t:|1781540475\n|-terastallize|p1a: Lugia|Ground\n|move|p1a: Lugia|Aeroblast|p2a: Hitmonlee\n|-supereffective|p2a: Hitmonlee\n|-damage|p2a: Hitmonlee|0 fnt\n|faint|p2a: Hitmonlee\n|\n|win|xValk_\n|raw|xValk_'s rating: 2138 &rarr; <strong>2158</strong><br />(+20 for winning)\n|raw|manwhocouldntcook's rating: 2146 &rarr; <strong>2126</strong><br />(-20 for losing)\n"

In [142]:
log.index("xValk_'s rating: ")

8974

In [147]:
for i in range(len(log)):
    if log.endswith("xValk_'s rating: ", i):
        print(i)

In [150]:
dict = {"player1": set(), "player2": set()}

In [151]:
dict

{'player1': set(), 'player2': set()}

In [152]:
dict["player1"]

set()